Importacion de librerias para manejo del sistema

In [22]:
import sys
import os

#Indicamos el workspace principal (dos niveles atrás)
sys.path.append(os.path.abspath('../../'))

%load_ext autoreload
%autoreload 2


In [33]:
#Manipulación y Análisis de Datos
import numpy as np
import pandas as pd
import pyreadstat
import plotly.express as px

#Funciones propias estadisticas
from src.estadisticas_pisa import puntuaciones_pisa_pais


# Carga de datos y exploracion

In [53]:
#TODO Mejora de la explicacion global

Para la lectura de datos, en vez de el metodo de pandas *read_sas* emplearemos [*pyreadstat*](https://github.com/Roche/pyreadstat/tree/master#reading-files) que presenta una mejor API para el manjo de datos SAS obteniendo el dataframe y sus metadatos

Realizamos la lectura de los metadatos, para obtener el nombre e informacion de las columnas 

In [34]:
#Realizamos la lectura de los datos de interes
df, meta = pyreadstat.read_sas7bdat('../../data/raw/CY08MSP_STU_QQQ.SAS7BDAT', metadataonly=True)

In [35]:
print(f'Numero de filas: {meta.number_rows} \nNumero de columnas: {meta.number_columns}')

Numero de filas: 613744 
Numero de columnas: 1278


Observamos la informacion contenida en las columnas (Solo los primeros datos)

In [36]:
#Pasamos el dicionario a una lista e iteramos sobre lso valores de la lista
first_values = list(meta.column_names_to_labels.items())[:7]
for clave,valor in first_values:
    print(f"{clave}: {valor}")

CNT: Country code 3-character
CNTRYID: Country Identifier
CNTSCHID: Intl. School ID
CNTSTUID: Intl. Student ID
CYC: PISA Assessment Cycle (2 digits + 2 character Assessment type - MS/FT)
NatCen: National Centre 6-digit Code
STRATUM: Stratum ID 5-character (cnt + original stratum ID)


# Datos resultados Matematicas PISA 22

In [37]:
columnas_lectura = [col for col in meta.column_names if 'PV' in col and 'MATH' in col]

In [38]:
columnas_id=['CNT','STRATUM','CNTSCHID']

In [39]:
print(f"Columnas encontradas: {len(columnas_lectura)}")
print(columnas_lectura)

Columnas encontradas: 10
['PV1MATH', 'PV2MATH', 'PV3MATH', 'PV4MATH', 'PV5MATH', 'PV6MATH', 'PV7MATH', 'PV8MATH', 'PV9MATH', 'PV10MATH']


In [40]:
#Añadimos el resto de columnas de interes para realizar el estudio : 
columnas_interes =  columnas_id+columnas_lectura+['W_FSTUWT']

In [41]:
df_pisa, meta_real = pyreadstat.read_sas7bdat(
    '../../data/raw/CY08MSP_STU_QQQ.SAS7BDAT', 
    usecols=columnas_interes
)

In [45]:
df_pisa.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 613744 entries, 0 to 613743
Data columns (total 14 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   CNT       613744 non-null  str    
 1   CNTSCHID  613744 non-null  float64
 2   STRATUM   613744 non-null  str    
 3   W_FSTUWT  613744 non-null  float64
 4   PV1MATH   613744 non-null  float64
 5   PV2MATH   613744 non-null  float64
 6   PV3MATH   613744 non-null  float64
 7   PV4MATH   613744 non-null  float64
 8   PV5MATH   613744 non-null  float64
 9   PV6MATH   613744 non-null  float64
 10  PV7MATH   613744 non-null  float64
 11  PV8MATH   613744 non-null  float64
 12  PV9MATH   613744 non-null  float64
 13  PV10MATH  613744 non-null  float64
dtypes: float64(12), str(2)
memory usage: 70.2 MB


In [46]:
display(df_pisa.head())
display(df_pisa.tail())

,CNT,CNTSCHID,STRATUM,W_FSTUWT,PV1MATH,PV2MATH,PV3MATH,PV4MATH,PV5MATH,PV6MATH,PV7MATH,PV8MATH,PV9MATH,PV10MATH
0,ALB,800282.0,ALB03,3.15874,179.583,201.334,222.248,194.943,222.755,213.100,221.021,293.710,226.912,254.745
1,ALB,800115.0,ALB03,4.34524,308.247,304.444,316.129,234.353,323.448,303.435,333.178,288.888,343.157,329.659
2,ALB,800242.0,ALB01,7.83860,267.514,343.743,327.157,290.771,316.227,309.451,323.337,307.508,356.244,295.401
3,ALB,800245.0,ALB08,8.49148,272.649,300.642,226.409,316.835,288.861,321.268,312.063,334.765,283.916,329.915
4,ALB,800285.0,ALB03,3.70957,435.473,491.174,478.365,454.352,491.031,504.919,468.401,463.913,470.627,499.257


,CNT,CNTSCHID,STRATUM,W_FSTUWT,PV1MATH,PV2MATH,PV3MATH,PV4MATH,PV5MATH,PV6MATH,PV7MATH,PV8MATH,PV9MATH,PV10MATH
613739,UZB,86000120.0,UZB26,69.40366,300.530,329.387,359.595,314.353,322.403,333.284,311.266,342.431,348.710,308.192
613740,UZB,86000140.0,UZB04,63.18910,265.080,257.582,327.787,311.439,267.167,323.318,273.206,268.522,296.958,273.324
613741,UZB,86000024.0,UZB10,65.47339,420.221,399.911,420.742,432.866,392.747,397.174,421.565,407.357,407.601,418.482
613742,UZB,86000174.0,UZB16,66.15595,343.129,355.853,296.233,335.538,372.287,315.866,335.566,405.146,313.752,307.822
613743,UZB,86000123.0,UZB10,65.47339,350.056,419.903,393.147,429.775,364.818,417.647,348.383,404.625,357.951,418.439


In [47]:
rendimiento_matematicas_pais_PISA= puntuaciones_pisa_pais(df_pisa,competencia='MATH')

In [48]:
#Este es el dataframe que contiene cada media de los PV de lectura independiente
rendimiento_matematicas_pais_PISA.head()

,CNT,media_math_pisa,varianza_imputacion
0,ALB,368.221720,0.369265
1,ARE,431.110477,0.127318
2,ARG,377.529029,0.174689
3,AUS,487.084254,0.120797
4,AUT,487.267499,0.171985


In [49]:
rendimiento_matematicas_pais_PISA.shape

(80, 3)

In [50]:
matematicas_top_cnt=rendimiento_matematicas_pais_PISA.sort_values(by='media_math_pisa',ascending=False)

In [52]:
fig=px.bar(matematicas_top_cnt,y='media_math_pisa',x='CNT')

In [ ]:
#TODO Nombre de los ejes del gráfico y explicacion

In [ ]:
fig.show()

In [21]:
ruta_salida = '../../data/intermediate/pisa22_math_pais_consolidado.parquet'
rendimiento_matematicas_pais_PISA.to_parquet(ruta_salida, index=False)